# 10 — Prefix Sum + Hash Map

## Learning goals
By the end, you should be able to:
- Build prefix sums and answer range-sum queries.
- Understand why `prefix[j] - prefix[i]` gives a subarray sum.
- Use a hash map to count or locate matching prefix sums.
- Solve subarray-sum problems, including arrays with negative numbers.
- Recognize when prefix sum + hashing is better than a sliding window.

**Core pattern:** A subarray sum can be expressed as the difference of two prefix sums.

## 1. What is a prefix sum?

A prefix sum stores the running total of an array.

For `nums = [2, 4, 1, 3]`, define:
- `prefix[0] = 0`
- `prefix[1] = 2`
- `prefix[2] = 2 + 4 = 6`
- `prefix[3] = 2 + 4 + 1 = 7`
- `prefix[4] = 2 + 4 + 1 + 3 = 10`

So the prefix array is `[0, 2, 6, 7, 10]`.

The leading zero makes range calculations simpler.

In [ ]:
def build_prefix(nums: list[int]) -> list[int]:
    prefix = [0]

    for num in nums:
        prefix.append(prefix[-1] + num)

    return prefix

nums = [2, 4, 1, 3]
prefix = build_prefix(nums)

print(prefix)  # [0, 2, 6, 7, 10]

## 2. Why does prefix subtraction work?

Suppose we want the sum of `nums[left:right + 1]`.

`prefix[right + 1]` contains everything from index `0` through `right`.

`prefix[left]` contains everything before `left`.

Subtracting removes the unwanted left portion:

\[
sum(left, right) = prefix[right + 1] - prefix[left]
\]

Example: sum from index `1` through `3` in `[2, 4, 1, 3]`:
- `prefix[4] = 10`
- `prefix[1] = 2`
- Answer = `10 - 2 = 8`.

In [ ]:
def range_sum(prefix: list[int], left: int, right: int) -> int:
    return prefix[right + 1] - prefix[left]

nums = [2, 4, 1, 3]
prefix = build_prefix(nums)
print(range_sum(prefix, 1, 3))  # 8

## 3. Prefix sum without an extra array

You do not always need to store every prefix sum. A running sum is enough when processing elements once.

Use an array when you need many later range-sum queries. Use a running sum when one pass is enough.

In [ ]:
def total_sum(nums: list[int]) -> int:
    running = 0

    for num in nums:
        running += num

    return running

print(total_sum([2, 4, 1, 3]))  # 10

## 4. Brute force subarray sums

A subarray is a contiguous part of an array.

A brute-force approach can choose every left and right boundary and calculate the sum. Recomputing each sum from scratch is slow: up to `O(n^3)` time.

Keeping a running sum for each fixed left boundary improves this to `O(n^2)` time.

In [ ]:
def count_subarrays_bruteforce(nums: list[int], target: int) -> int:
    count = 0

    for left in range(len(nums)):
        current_sum = 0

        for right in range(left, len(nums)):
            current_sum += nums[right]

            if current_sum == target:
                count += 1

    return count

print(count_subarrays_bruteforce([1, 1, 1], 2))  # 2

## 5. The key equation

Let `prefix` be the sum of all elements before the current position, and let `target` be the required subarray sum.

We need an earlier prefix sum `old_prefix` such that:

\[
prefix - old\_prefix = target
\]

Rearrange:

\[
old\_prefix = prefix - target
\]

**This is the key insight:** At each position, look up `current_prefix - target` in a hash map.

The map stores how many times each prefix sum has appeared.

## 6. Count subarrays with sum equal to k

Algorithm:
1. Start with `prefix_sum = 0`.
2. Initialize `freq = {0: 1}`. This represents the empty prefix and handles subarrays starting at index `0`.
3. For each number, update the running prefix sum.
4. Add `freq[prefix_sum - k]` to the answer.
5. Record the current prefix sum in the map.

Lookup must happen before recording the current prefix. Otherwise, the current prefix could match itself and incorrectly represent an empty subarray when `k = 0`.

In [ ]:
def subarray_sum(nums: list[int], k: int) -> int:
    prefix_sum = 0
    count = 0
    freq = {0: 1}

    for num in nums:
        prefix_sum += num

        needed = prefix_sum - k
        count += freq.get(needed, 0)

        freq[prefix_sum] = freq.get(prefix_sum, 0) + 1

    return count

print(subarray_sum([1, 1, 1], 2))       # 2
print(subarray_sum([1, 2, 3], 3))       # 2
print(subarray_sum([1, -1, 0], 0))      # 3
print(subarray_sum([3, 4, -7, 1, 3, 3, 1, -4], 7))  # 4

## 7. Dry run — `nums = [1, 2, 3]`, `k = 3`

Initial state:
- `prefix_sum = 0`
- `freq = {0: 1}`
- `count = 0`

| Number | Prefix | Needed (`prefix - k`) | Matches before update | Count |
|---:|---:|---:|---:|---:|
| 1 | 1 | -2 | 0 | 0 |
| 2 | 3 | 0 | 1 | 1 |
| 3 | 6 | 3 | 1 | 2 |

Matching subarrays:
- `[1, 2]`
- `[3]`

Answer: `2`.

Notice that the map stores prefix-sum frequencies, not array values.

## 8. Why this works with negative numbers

A normal sliding window often relies on the sum moving in one direction when the window grows or shrinks. Negative values break that assumption.

Prefix sum + hash map does not need positive numbers. It uses an equality between two prefix sums, so negative values are naturally supported.

In [ ]:
nums = [1, -1, 0]
print(subarray_sum(nums, 0))  # 3

# The zero-sum subarrays are:
# [1, -1], [0], and [1, -1, 0]

## 9. Find whether a subarray with sum k exists

If you only need a boolean, store previously seen prefix sums in a set.

At each position, if `prefix_sum - k` has appeared, a matching subarray exists.

In [ ]:
def has_subarray_sum(nums: list[int], k: int) -> bool:
    prefix_sum = 0
    seen = {0}

    for num in nums:
        prefix_sum += num

        if prefix_sum - k in seen:
            return True

        seen.add(prefix_sum)

    return False

print(has_subarray_sum([1, 2, 3], 5))  # True
print(has_subarray_sum([1, 2, 3], 7))  # False

## 10. Return one matching subarray's indices

Store the first index at which each prefix sum appears.

If current prefix sum is `prefix_sum`, and `prefix_sum - k` was first seen at index `j`, the matching subarray starts at `j + 1` and ends at the current index.

We use prefix index `-1` for the empty prefix before the array begins.

In [ ]:
def find_subarray_sum_indices(nums: list[int], k: int):
    first_index = {0: -1}
    prefix_sum = 0

    for i, num in enumerate(nums):
        prefix_sum += num
        needed = prefix_sum - k

        if needed in first_index:
            return first_index[needed] + 1, i

        if prefix_sum not in first_index:
            first_index[prefix_sum] = i

    return None

print(find_subarray_sum_indices([2, 4, -1, 3], 3))

## 11. Longest subarray with sum k

To maximize the length, keep the **first** index where each prefix sum appeared.

If the current prefix is `P`, an earlier prefix `P - k` creates a subarray with sum `k`. The earliest such index gives the longest subarray ending here.

Do not overwrite an earlier index for the same prefix sum.

In [ ]:
def longest_subarray_sum_k(nums: list[int], k: int) -> int:
    first_index = {0: -1}
    prefix_sum = 0
    best = 0

    for i, num in enumerate(nums):
        prefix_sum += num
        needed = prefix_sum - k

        if needed in first_index:
            best = max(best, i - first_index[needed])

        if prefix_sum not in first_index:
            first_index[prefix_sum] = i

    return best

print(longest_subarray_sum_k([1, -1, 5, -2, 3], 3))  # 4
print(longest_subarray_sum_k([-2, -1, 2, 1], 1))     # 2

## 12. Count subarrays with sum zero

Set `k = 0` in the main equation.

We need an earlier prefix sum equal to the current prefix sum. Each repeated prefix sum identifies a zero-sum subarray.

In [ ]:
def count_zero_sum_subarrays(nums: list[int]) -> int:
    return subarray_sum(nums, 0)

print(count_zero_sum_subarrays([1, -1, 0]))  # 3
print(count_zero_sum_subarrays([1, 2, 3]))  # 0

## 13. Count subarrays with sum divisible by k

For integer prefix sums `P1` and `P2`, the subarray sum is divisible by `k` when:

\[
(P2 - P1) \bmod k = 0
\]

That happens when both prefix sums have the same remainder modulo `k`.

So count earlier occurrences of the current remainder.

Python's modulo handles negative numbers consistently. The problem normally guarantees `k != 0`.

In [ ]:
def subarrays_div_by_k(nums: list[int], k: int) -> int:
    remainder_count = {0: 1}
    prefix_sum = 0
    answer = 0

    for num in nums:
        prefix_sum += num
        remainder = prefix_sum % k

        answer += remainder_count.get(remainder, 0)
        remainder_count[remainder] = remainder_count.get(remainder, 0) + 1

    return answer

print(subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5))  # 7

## 14. Prefix sum + hash map vs sliding window

| Situation | Likely pattern |
|---|---|
| Count subarrays with exact sum, values may be negative | Prefix sum + hash map |
| Longest subarray with exact sum, values may be negative | Prefix sum + first-index map |
| Count subarrays whose sum is divisible by k | Prefix remainder + frequency map |
| Positive numbers only and a sum threshold | Sliding window may apply |
| Many static range-sum queries | Prefix-sum array |

Do not choose sliding window just because the problem says “subarray.” Check whether its assumptions are satisfied.

## 15. Common mistakes

1. Forgetting `freq = {0: 1}`. This misses valid subarrays starting at index `0`.
2. Storing array values instead of prefix sums.
3. Updating the map before checking the needed prefix.
4. Using a set when you need to count multiple subarrays; use a frequency map.
5. Overwriting the earliest index when finding the longest subarray.
6. Assuming sliding window works with negative numbers.
7. Confusing a subarray (contiguous) with a subsequence (not necessarily contiguous).
8. Forgetting the special case `k = 0` in divisibility problems.

## 16. Interview recognition checklist

Look for:
- “number of subarrays whose sum equals k”
- “longest subarray with sum k”
- “zero-sum subarray”
- “sum divisible by k”
- “equal number of 0s and 1s” (often transform values and use prefix sums)
- “range sum queries”

Ask:
1. Is the target about a contiguous subarray?
2. Can values be negative?
3. Do I need to count, find one, or maximize length?
4. Should the map store frequencies or first indices?
5. What initial prefix state handles a subarray starting at index zero?

**Core equation:** `previous_prefix = current_prefix - target`.

## 17. Practice questions — solve in this order

### Easy
1. [1480. Running Sum of 1d Array](https://leetcode.com/problems/running-sum-of-1d-array/)
2. [303. Range Sum Query — Immutable](https://leetcode.com/problems/range-sum-query-immutable/)

### Medium
3. [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/)
4. [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/)
5. [974. Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/)
6. [523. Continuous Subarray Sum](https://leetcode.com/problems/continuous-subarray-sum/)
7. [325. Maximum Size Subarray Sum Equals k](https://leetcode.com/problems/maximum-size-subarray-sum-equals-k/) — Premium

For each problem:
- State the brute-force approach.
- Define the prefix sum precisely.
- Derive the equation before coding.
- Decide whether the map stores counts, first indices, or remainders.
- Test negative numbers, zeros, repeated prefix sums, and subarrays starting at index zero.
- State time and space complexity.

## 18. Reusable templates

### Count subarrays with sum k
```python
def count_subarrays_sum_k(nums: list[int], k: int) -> int:
    prefix = 0
    answer = 0
    freq = {0: 1}

    for num in nums:
        prefix += num
        answer += freq.get(prefix - k, 0)
        freq[prefix] = freq.get(prefix, 0) + 1

    return answer
```

### Longest subarray with sum k
```python
def longest_subarray_sum_k(nums: list[int], k: int) -> int:
    prefix = 0
    best = 0
    first = {0: -1}

    for i, num in enumerate(nums):
        prefix += num

        if prefix - k in first:
            best = max(best, i - first[prefix - k])

        if prefix not in first:
            first[prefix] = i

    return best
```

### Final mental model
- **Frequency map:** How many earlier prefixes match?
- **First-index map:** How far back can I go?
- **Set:** Has a matching prefix appeared?

This completes the foundation of Prefix Sum + Hash Map.